In [ ]:
import hashlib
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from lancedb.rerankers import ColbertReranker
import torch
import ollama
import os
import json
import subprocess
from tqdm.notebook import tqdm

EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
MAX_TOKENS = 2000
OLLAMA_MODEL_NAME= "chunker_full_doc"
# CHUNKS_WITH_METADATA_FILE_NAME = "preprocessed_chunks/anthropic_control_chunks_with_metadata.json"
doc_slice_radius = 4

CHUNKS_WITH_METADATA_FILE_NAME = f"preprocessed_chunks/ablation_doc_slice_radius_{doc_slice_radius}.json"

INPUT_DIR = "split_documents"

study_names = [f for f in os.listdir(INPUT_DIR) if f.endswith('.json')]
processed_chunks=[]
try:
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        processed_chunks = json.load(f)
except FileNotFoundError:
    print(f"No existing {CHUNKS_WITH_METADATA_FILE_NAME} file found, starting fresh.")
    

chunks_with_metadata = processed_chunks.copy()
processed_studies = set(chunk["document"] for chunk in processed_chunks)

study_names = [f for f in study_names if f not in processed_studies]
print(f"Found {len(processed_studies)} studies which are already processed.\nStudies which STILL need to be processed: {len(study_names)}:\n{study_names}...")

# k=4 --> 28m11s
# k=3 --> 22m41s
# k=2 --> 17m28s
# k=1 --> 12m08s
# k=0 --> 7m28s

/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in ColPaliEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in SigLipEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


No existing preprocessed_chunks/ablation_doc_slice_radius_4.json file found, starting fresh.
Found 0 studies which are already processed.
Studies which STILL need to be processed: 25:
['A_Conceptual_Framework_and_Recommendations_for_Open_Data_and_Artifacts_in_Empirical_Software_Engineering.pdf.json', 'A_Feature_Fusion_Based_Indicator_for_Training-Free_Neural_Architecture_Search.pdf.json', 'A_Hybrid_Gaze_Distance_Estimation_via_Cross-Reference_of_Vergence_and_Depth.pdf.json', 'A_Resource_Allocation_Model_Based_on_Trust_Evaluation_in_Multi-Cloud_Environments.pdf.json', 'Electron_Paramagnetic_Resonance_Study_on_28Si_Single_Crystal_for_the_Future_Realization_of_the_Kilogram.pdf.json', 'Probabilistic_Artificial_Neural_Network_for_Line-Edge-Roughness-Induced_Random_Variation_in_FinFET.pdf.json', 'Quantitative_Evaluation_of_Line-Edge_Roughness_in_Various_FinFET_Structures_Bayesian_Neural_Network_With_Automatic_Model_Selection.pdf.json', 'Realization_of_a_Rubidium_Atomic_Frequency_Standard_Wit

In [2]:
# context_length = max(16_000, doc_slice_radius * 2 + 2 * MAX_TOKENS) # We need to reserve space for the chunk itself (twice, the context contains the chunk itself)
context_length = doc_slice_radius * 2 + 2 * MAX_TOKENS


for source in tqdm(study_names, desc="Chunking documents..."):        
	try:
		with open(os.path.join(INPUT_DIR, source), "r", encoding="utf-8") as f:
			chunks = json.load(f)
	except FileNotFoundError:
		print(f"No existing {CHUNKS_WITH_METADATA_FILE_NAME} file found, starting fresh.")

	for chunk_index, chunk in enumerate(tqdm(chunks, desc=f"Adding context for chunks of {source[:20]}...", leave=False)):    
		doc_slice = ""	

		start_index_original = chunk_index - doc_slice_radius
		start_index_truncated = max(0, start_index_original) # Avoid index out of bounds

		end_index_original = chunk_index + doc_slice_radius
		end_index_truncated = min(len(chunks)-1, end_index_original)

		if start_index_original < 0: # We are at the start of the document, so we need to add more chunks at the end
			end_index_truncated = min(len(chunks)-1, end_index_truncated + abs(start_index_original))
		if end_index_original > len(chunks)-1: # We are at the end of the document, so we need to add more chunks at the start
			start_index_truncated = max(0, start_index_truncated - abs(end_index_original - end_index_truncated))

		for i in range(start_index_truncated, end_index_truncated + 1):
			doc_slice += " " + chunks[i]["text"]

		doc_slice = "FULL DOCUMENT:\n" + doc_slice
		ollama_prompt = f"CHUNK:\n{chunks[chunk_index]['text']}"
		history =  [{'role': 'user', 'content': doc_slice}, {'role': 'user', 'content': ollama_prompt}]

		response = ollama.chat(
			model=OLLAMA_MODEL_NAME,
			messages=history,
		)
		context = response['message']['content']
		text_to_embed = context + "\n\n" + chunks[chunk_index]['text'] 
  
		id = hashlib.sha256(chunks[chunk_index]['text'].encode()).hexdigest()
		chunks_with_metadata.append({'text': text_to_embed, 'original_text':chunks[chunk_index]['text'], 'context':context, 'document':source, 'id': id})

subprocess.run(["ollama", "stop", OLLAMA_MODEL_NAME], check=True)
		

Chunking documents...:   0%|          | 0/25 [00:00<?, ?it/s]

Adding context for chunks of A_Conceptual_Framewo...:   0%|          | 0/21 [00:00<?, ?it/s]

Adding context for chunks of A_Feature_Fusion_Bas...:   0%|          | 0/26 [00:00<?, ?it/s]

Adding context for chunks of A_Hybrid_Gaze_Distan...:   0%|          | 0/14 [00:00<?, ?it/s]

Adding context for chunks of A_Resource_Allocatio...:   0%|          | 0/18 [00:00<?, ?it/s]

...

Adding context for chunks of srep03578.pdf.json...:   0%|          | 0/10 [00:00<?, ?it/s]

Adding context for chunks of srep04487.pdf.json...:   0%|          | 0/11 [00:00<?, ?it/s]

Adding context for chunks of srep05215.pdf.json...:   0%|          | 0/15 [00:00<?, ?it/s]

Adding context for chunks of srep45325.pdf.json...:   0%|          | 0/8 [00:00<?, ?it/s]

CompletedProcess(args=['ollama', 'stop', 'chunker_full_doc'], returncode=0)

In [3]:
chunks_with_metadata[0]

{'text': 'This chunk introduces the research project, the Gander case, and its focus on deriving advice for open science practices in empirical software engineering, specifically addressing the balance between openness, integrity, and secrecy concerns.\n\nPer Runeson per.runeson@cs.lth.se Lund University Lund, Sweden\nEmma Söderberg emma.soderberg@cs.lth.se Lund University Lund, Sweden\nMartin Höst martin.host@mau.se Malmö University Malmö, Sweden',
 'original_text': 'Per Runeson per.runeson@cs.lth.se Lund University Lund, Sweden\nEmma Söderberg emma.soderberg@cs.lth.se Lund University Lund, Sweden\nMartin Höst martin.host@mau.se Malmö University Malmö, Sweden',
 'context': 'This chunk introduces the research project, the Gander case, and its focus on deriving advice for open science practices in empirical software engineering, specifically addressing the balance between openness, integrity, and secrecy concerns.',
 'document': 'A_Conceptual_Framework_and_Recommendations_for_Op

In [4]:
# Save the the processed chunks in case VectorDB upload goes wrong.
# Luckily since this is a notebook, if the chunking is interrupted, we can still save the partial results here.
# Append new chunks to the existing file if it exists, otherwise create it
if os.path.exists(CHUNKS_WITH_METADATA_FILE_NAME):
    print(f"Appending to existing {CHUNKS_WITH_METADATA_FILE_NAME} file.")
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    # Avoid duplicate entries by id
    existing_ids = {chunk['id'] for chunk in existing_data}
    new_chunks = [chunk for chunk in chunks_with_metadata if chunk['id'] not in existing_ids]
    chunks_with_metadata = existing_data + new_chunks

with open(CHUNKS_WITH_METADATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

print(f"Results saved to {CHUNKS_WITH_METADATA_FILE_NAME}")

Results saved to preprocessed_chunks/ablation_doc_slice_radius_4.json


In [5]:

from devtools import debug
registry = get_registry()
hf = registry.get("huggingface").create(name=EMBEDDING_MODEL_NAME, trust_remote_code=True, device="cuda" if torch.cuda.is_available() else "cpu")


# Define model
class MyDocument(LanceModel):
    text: str 
    vector: Vector(hf.ndims()) = hf.VectorField()
    original_text: str = hf.SourceField()
    context: str
    document: str
    id: str  # Unique identifier for the chunk


db = lancedb.connect("./db")
table_name = f"ablation_doc_slice_radius_{doc_slice_radius}"
db.create_table(table_name, schema=MyDocument, mode="overwrite") # Uncomment this line when running this cell for the first time
table = db.open_table(table_name)

# Upload in batches with progress bar
with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
    chunks_with_metadata = json.load(f)
    debug(chunks_with_metadata[0])

batch_size = 100
for i in tqdm(range(0, len(chunks_with_metadata), batch_size), desc="Uploading chunks to VectorDB"):
    batch = chunks_with_metadata[i:i+batch_size]
    table.add(batch)

table.create_scalar_index("id", replace=True) # Index based on the chunk's id, used to manually prevent duplicates

reranker = ColbertReranker()
table.create_fts_index("text", replace=True) # Used by the reranker as well as the hybrid search's BM25 index
table.wait_for_index(["text_idx"])  # Wait for the indexing to finish

<All keys matched successfully>


/tmp/ipykernel_177915/2100205930.py:24 <module>
    chunks_with_metadata[0]: {
        'text': (
            'This chunk introduces the research project, the Gander case, and its focus on deriving advice for open sc'
            'ience practices in empirical software engineering, specifically addressing the balance between openness, '
            'integrity, and secrecy concerns.\n'
            '\n'
            'Per Runeson per.runeson@cs.lth.se Lund University Lund, Sweden\n'
            'Emma Söderberg emma.soderberg@cs.lth.se Lund University Lund, Sweden\n'
            'Martin Höst martin.host@mau.se Malmö University Malmö, Sweden'
        ),
        'original_text': (
            'Per Runeson per.runeson@cs.lth.se Lund University Lund, Sweden\n'
            'Emma Söderberg emma.soderberg@cs.lth.se Lund University Lund, Sweden\n'
            'Martin Höst martin.host@mau.se Malmö University Malmö, Sweden'
        ),
        'context': (
            'This chunk introduces the 

Uploading chunks to VectorDB:   0%|          | 0/4 [00:00<?, ?it/s]

<All keys matched successfully>
<All keys matched successfully>
<All keys matched successfully>
<All keys matched successfully>


Loading ColBERTRanker model colbert-ir/colbertv2.0 (this message can be suppressed by setting verbose=0)
No device set
Using device cuda
No dtype set
Using dtype torch.float32
Loading model colbert-ir/colbertv2.0, this might take a while...
Linear Dim set to: 128 for downcasting
